In [0]:
# Camada Bronze - Ingestão dos CSVs brutos

In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOGO = "workspace"
SCHEMA_BRONZE = "bronze"
CAMINHO_VOLUME = "/Volumes/workspace/inputs/cinedata"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;

In [0]:
# Mapeamento entre o arquivo de origem no volume e a tabela de destino na bronze

mapa_arquivos_tabelas = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews",
}

In [0]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(f"{CAMINHO_VOLUME}/movies_info_TMDB_IMDB.csv")
)

print(f"Registros lidos: {df_raw.count()}")
display(df_raw.limit(5))

Registros lidos: 106596


id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.


In [0]:
# Ingestão dos CSVs na camada bronze

for nome_arquivo, nome_tabela in mapa_arquivos_tabelas.items():
    # Leitura:
    df_raw = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(f"{CAMINHO_VOLUME}/{nome_arquivo}")
    )

    df_bronze = df_raw.withColumn("ingestion_datetime", F.current_timestamp())

    # Escrita:
    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(f"{CATALOGO}.{SCHEMA_BRONZE}.{nome_tabela}")
    )

    print(f"{nome_arquivo} -> {SCHEMA_BRONZE}.{nome_tabela}: {df_bronze.count()} registros")

movies_info_TMDB_IMDB.csv -> bronze.tb_movies_info: 106596 registros
movies_financials_IMDB_TMDB.csv -> bronze.tb_movies_financials: 106165 registros
movies_metrics_IMDB_TMDB.csv -> bronze.tb_movies_metrics: 103072 registros
credits_and_tags_IMDB_TMDB.csv -> bronze.tb_credits_and_tags: 105608 registros
movies_reviews.csv -> bronze.tb_movies_reviews: 32412 registros


In [0]:
# Validação

for nome_tabela in mapa_arquivos_tabelas.values():
    df_validacao = spark.read.table(f"{CATALOGO}.{SCHEMA_BRONZE}.{nome_tabela}")
    print(f"{nome_tabela}: {df_validacao.count()} registros | "
          f"{df_validacao.select('ingestion_datetime').distinct().count()} lote(s)")

tb_movies_info: 213192 registros | 2 lote(s)
tb_movies_financials: 212330 registros | 2 lote(s)
tb_movies_metrics: 206144 registros | 2 lote(s)
tb_credits_and_tags: 211216 registros | 2 lote(s)
tb_movies_reviews: 64824 registros | 2 lote(s)


In [0]:
# Parâmetros do notebook: período de consulta da cotação 
# Padrão: últimos 7 dias corridos

from datetime import date, timedelta

data_fim_padrao = date.today()
data_inicio_padrao = data_fim_padrao - timedelta(days=7)

dbutils.widgets.text("data_inicio", data_inicio_padrao.strftime("%m-%d-%Y"), "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", data_fim_padrao.strftime("%m-%d-%Y"), "Data fim (MM-DD-AAAA)")

In [0]:
import requests

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

url_api = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'"
    f"&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

# Leitura:
resposta = requests.get(url_api, timeout=30)
resposta.raise_for_status()  # interrompe o pipeline se a API retornar erro HTTP

dados_cotacao = resposta.json()["value"]

# Sem cotações no período
if not dados_cotacao:
    raise ValueError(f"A API não retornou cotações entre {data_inicio_formatada} e {data_fim_formatada}.")

print(f"Cotações retornadas: {len(dados_cotacao)}")
print(dados_cotacao[:2])

Cotações retornadas: 6
[{'cotacaoCompra': 5.1803, 'dataHoraCotacao': '2026-09-30 13:11:44.783262'}, {'cotacaoCompra': 5.2073, 'dataHoraCotacao': '2026-10-01 13:10:35.4469'}]


In [0]:
df_raw = spark.createDataFrame(dados_cotacao)

df_bronze = df_raw.withColumn("ingestion_datetime", F.current_timestamp())

# Escrita:
(
    df_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{CATALOGO}.{SCHEMA_BRONZE}.tb_cotacao_dolar")
)

display(spark.read.table(f"{CATALOGO}.{SCHEMA_BRONZE}.tb_cotacao_dolar"))

cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.1803,2026-09-30 13:11:44.783262,2026-10-07T20:08:40.917Z
5.2073,2026-10-01 13:10:35.4469,2026-10-07T20:08:40.917Z
5.2232,2026-10-02 13:03:16.256632,2026-10-07T20:08:40.917Z
4.9853,2026-10-05 13:07:36.558602,2026-10-07T20:08:40.917Z
4.9692,2026-10-06 13:03:21.267287,2026-10-07T20:08:40.917Z
4.9929,2026-10-07 13:05:23.249063,2026-10-07T20:08:40.917Z
